In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [3]:
df = pd.read_csv('housing.csv')

In [4]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [5]:
df['income_cat'] = pd.cut(df['median_house_value'], 
                          bins = [0, 1.5, 3, 4.5, 6, np.inf], 
                          labels = [1, 2, 3, 4, 5])
split = StratifiedShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 42)
for train_index, test_index in split.split(df, df['income_cat']):
    train_data = df.loc[train_index].drop('income_cat', axis = 1)
    test_data = df.loc[test_index].drop('income_cat', axis = 1)

In [6]:
housing = train_data.copy()
housing_labels = housing['median_house_value']
housing = housing.drop('median_house_value', axis = 1)

In [7]:
num_attributes = housing.drop('ocean_proximity', axis = 1).columns.tolist()
cat_attributes = ['ocean_proximity']

In [8]:
#PipeLine

#num_pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scalar', StandardScaler()),
])

#cat_pipeline
cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [9]:
full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_attributes),
    ('cat', cat_pipeline, cat_attributes)
])

In [10]:
housing_prepared = full_pipeline.fit_transform(housing)
housing_prepared

array([[ 0.77738183, -0.86037145, -0.44938232, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.86236206, -0.88850205, -0.68766751, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.98233415, -0.82286399, -1.08480949, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.5974237 , -0.85099458,  0.34490165, ...,  0.        ,
         0.        ,  1.        ],
       [-1.23215067,  0.89779089, -1.0053811 , ...,  0.        ,
         1.        ,  0.        ],
       [ 0.62241788, -0.71971846,  1.21861402, ...,  0.        ,
         0.        ,  0.        ]], shape=(16512, 13))

In [11]:
df['ocean_proximity'].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [12]:
housing_prepared = pd.DataFrame(housing_prepared, columns = num_attributes+['<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'NEAR BAY', 'ISLAND'], index = housing.index)
housing_prepared

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,<1H OCEAN,INLAND,NEAR OCEAN,NEAR BAY,ISLAND
11604,0.777382,-0.860371,-0.449382,0.015122,-0.254032,0.069514,-0.249947,1.008984,1.0,0.0,0.0,0.0,0.0
10897,0.862362,-0.888502,-0.687668,-0.026555,0.469354,0.101068,0.449227,-0.764341,1.0,0.0,0.0,0.0,0.0
12076,0.982334,-0.822864,-1.084809,2.288107,2.476631,2.164693,2.410066,-0.226004,0.0,1.0,0.0,0.0,0.0
13277,0.962339,-0.719718,0.424330,0.095727,0.294496,0.360711,0.349345,-0.432680,0.0,1.0,0.0,0.0,0.0
308,-1.307133,1.005625,1.774613,-0.239519,-0.155824,-0.224388,-0.157950,-0.324464,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12528,-0.962213,1.376011,0.503758,-0.507899,-0.189358,-0.392074,-0.113266,-1.038466,0.0,1.0,0.0,0.0,0.0
5642,0.627417,-0.883814,0.583187,-0.076018,-0.153429,-0.280283,-0.108009,0.335905,0.0,0.0,0.0,0.0,1.0
8763,0.597424,-0.850995,0.344902,1.111540,0.608283,0.540118,0.656876,1.574111,0.0,0.0,0.0,0.0,1.0
941,-1.232151,0.897791,-1.005381,0.837207,0.766374,1.456984,0.977549,0.634659,0.0,0.0,0.0,1.0,0.0


In [13]:
#Training Algo
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [14]:
#Decision_Tree
dt_reg = DecisionTreeRegressor()
dt_reg.fit(housing_prepared, housing_labels)
dt_predicts = dt_reg.predict(housing_prepared)
dt_rmse = root_mean_squared_error(dt_predicts, housing_labels)
print(f"The RMSE using Decision Tree Regression is - {dt_rmse}")

The RMSE using Decision Tree Regression is - 0.0


In [15]:
#Random_Forest
rf_reg = RandomForestRegressor()
rf_reg.fit(housing_prepared, housing_labels)
rf_predicts = rf_reg.predict(housing_prepared)
rf_rmse = root_mean_squared_error(rf_predicts, housing_labels)
print(f"The RMSE using Random_Forest Regression is - {rf_rmse}")

The RMSE using Random_Forest Regression is - 18066.47292453042


In [16]:
#Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)
lin_predicts = lin_reg.predict(housing_prepared)
lin_rmse = root_mean_squared_error(lin_predicts, housing_labels)
print(f"The RMSE using Linear Regression is - {lin_rmse}")

The RMSE using Linear Regression is - 67983.1570343809


In [24]:
#Using Cross Validation -
'''Instead of training the model once and evaluating on a holdout set, 
k-fold cross-validation splits the training data into k folds (typically 10), trains the model on k-1 folds, 
and validates it on the remaining fold. This process repeats k times.'''

from sklearn.model_selection import cross_val_score


In [28]:
#Decision_Tree
dt_reg = DecisionTreeRegressor()
dt_reg.fit(housing_prepared, housing_labels)
dt_predicts = dt_reg.predict(housing_prepared)
dt_rmse = -cross_val_score(dt_reg, housing_prepared, housing_labels, scoring='neg_root_mean_squared_error', cv = 10)
print(f"The RMSE using Decision Tree Regression is - {dt_rmse}")
print(pd.Series(dt_rmse).describe())

The RMSE using Decision Tree Regression is - [64612.48265108 74056.35277682 66463.17405107 68568.03991949
 69136.39172924 65898.36882173 68100.54742854 67995.31643771
 66287.10708367 68526.6952367 ]
count       10.000000
mean     67964.447614
std       2576.740574
min      64612.482651
25%      66331.123826
50%      68047.931933
75%      68557.703749
max      74056.352777
dtype: float64


In [29]:
#Random_Forest
rf_reg = RandomForestRegressor()
rf_reg.fit(housing_prepared, housing_labels)
rf_predicts = rf_reg.predict(housing_prepared)
rf_rmse = -cross_val_score(dt_reg, housing_prepared, housing_labels, scoring='neg_root_mean_squared_error', cv = 10)
print(f"The RMSE using Random_Forest Regression is - {rf_rmse}")
print(pd.Series(rf_rmse).describe())

The RMSE using Random_Forest Regression is - [65363.93917692 72292.01388192 66201.86967931 67651.84081481
 68261.96997189 65226.79377252 67875.90343258 69696.66702991
 64686.39622155 69016.10353549]
count       10.000000
mean     67627.349752
std       2357.065540
min      64686.396222
25%      65573.421803
50%      67763.872124
75%      68827.570145
max      72292.013882
dtype: float64


In [30]:
#Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)
lin_predicts = lin_reg.predict(housing_prepared)
lin_rmse = -cross_val_score(dt_reg, housing_prepared, housing_labels, scoring='neg_root_mean_squared_error', cv = 10)
print(f"The RMSE using Linear Regression is - {lin_rmse}")
print(pd.Series(lin_rmse).describe())

The RMSE using Linear Regression is - [66106.61003991 73669.11034518 67915.99823064 66530.89526854
 68159.0374385  65238.4727565  69275.70994169 70368.54168964
 63985.88524354 69676.51084617]
count       10.000000
mean     68092.677180
std       2820.869235
min      63985.885244
25%      66212.681347
50%      68037.517835
75%      69576.310620
max      73669.110345
dtype: float64
